**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 🎯 OpenAI API 소개 및 실습

## Part 1 | Session 4: OpenAI API, Chat Completions, 스트리밍, 멀티턴 대화

---

### 📋 학습 목표

- 🔹 OpenAI API의 기본 구조와 설정 방법을 이해합니다
- 🔹 Chat Completions API로 모델과 대화할 수 있습니다
- 🔹 시스템 프롬프트와 역할(role)의 개념을 이해합니다
- 🔹 temperature, top_p, max_tokens 등 주요 파라미터를 조절합니다
- 🔹 스트리밍(streaming) 응답을 처리합니다
- 🔹 멀티턴 대화와 JSON 모드를 구현합니다

### 📦 필요 라이브러리

```
openai, python-dotenv
```

### ⚠️ 사전 준비

- OpenAI API Key가 필요합니다
- https://platform.openai.com/api-keys 에서 발급 가능

### ⏱️ 예상 소요 시간: 약 90분

---

---

## 🎯 OpenAI API 소개 및 설정

### API Key 설정 방법

API Key를 안전하게 관리하기 위해 **환경변수** 또는 **.env 파일**을 사용합니다.

#### 방법 1: .env 파일 생성 (권장)

프로젝트 루트에 `.env` 파일을 생성하고 다음과 같이 작성합니다:

```
OPENAI_API_KEY=sk-your-api-key-here
```

⚠️ **주의**: `.env` 파일은 절대 git에 커밋하지 마세요! `.gitignore`에 추가하세요.

#### 방법 2: 터미널에서 환경변수 설정

```bash
export OPENAI_API_KEY="sk-your-api-key-here"
```

### OpenAI 주요 모델

| 모델 | 용도 | 특징 |
|------|------|------|
| gpt-4o | 범용 최신 모델 | 빠르고 저렴, 멀티모달 |
| gpt-4o-mini | 경량 모델 | 매우 저렴, 간단한 작업 |
| gpt-4-turbo | 고성능 모델 | 복잡한 추론 |
| o1 / o3 | 추론 모델 | 수학/코딩 강점 |

---

In [ ]:
# 필요 라이브러리 설치
# !pip install openai python-dotenv

import os
from dotenv import load_dotenv
from openai import OpenAI

# .env 파일에서 환경변수 로드
load_dotenv()

# API Key 확인
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ OpenAI API Key가 설정되었습니다!")
    print(f"   Key 앞 8자: {api_key[:8]}...")
else:
    print("❌ API Key가 설정되지 않았습니다!")
    print("   다음 중 하나의 방법으로 설정해주세요:")
    print("   1. .env 파일에 OPENAI_API_KEY=sk-... 추가")
    print("   2. 아래 셀에서 직접 입력")

# OpenAI 클라이언트 생성
client = OpenAI(api_key=api_key)
print("\n✅ OpenAI 클라이언트 생성 완료!")

In [ ]:
# (선택) API Key가 없는 경우 직접 입력
# 아래 주석을 해제하고 API Key를 입력하세요

# import getpass
# api_key = getpass.getpass("OpenAI API Key를 입력하세요: ")
# os.environ["OPENAI_API_KEY"] = api_key
# client = OpenAI(api_key=api_key)
# print("✅ API Key가 설정되었습니다!")

---

## 1️⃣ Chat Completions API 기본 사용법

### API 호출 구조

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",       # 사용할 모델
    messages=[                   # 대화 메시지 리스트
        {"role": "system", "content": "..."},  # 시스템 프롬프트
        {"role": "user", "content": "..."},    # 사용자 메시지
    ],
    temperature=0.7,             # 창의성 조절 (0~2)
    max_tokens=500,              # 최대 출력 토큰 수
)
```

### 메시지 역할(Role)

| 역할 | 설명 |
|------|------|
| `system` | 모델의 행동 지침 설정 (예: "당신은 전문 번역가입니다") |
| `user` | 사용자의 질문/요청 |
| `assistant` | 모델의 이전 응답 (멀티턴 대화에서 사용) |

---

In [ ]:
# Chat Completions API 기본 호출
print("=" * 60)
print("💬 Chat Completions API 기본 호출")
print("=" * 60)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "LLM 파인튜닝이 뭔지 3줄로 설명해줘."}
    ],
    temperature=0.7,
    max_tokens=300,
)

# 응답 내용 출력
answer = response.choices[0].message.content
print(f"\n🤖 모델 응답:\n{answer}")

# 응답 메타데이터 확인
print(f"\n📊 응답 메타데이터:")
print(f"   모델: {response.model}")
print(f"   입력 토큰: {response.usage.prompt_tokens}")
print(f"   출력 토큰: {response.usage.completion_tokens}")
print(f"   총 토큰: {response.usage.total_tokens}")
print(f"   종료 이유: {response.choices[0].finish_reason}")

In [ ]:
# 응답 객체 구조 살펴보기
print("=" * 60)
print("🔍 응답 객체(Response Object) 구조")
print("=" * 60)

print(f"\n📦 전체 응답 타입: {type(response).__name__}")
print(f"\n📌 주요 필드:")
print(f"   response.id           = {response.id}")
print(f"   response.model        = {response.model}")
print(f"   response.created      = {response.created}")
print(f"   response.choices      = [{len(response.choices)}개의 선택지]")
print(f"   response.usage        = {response.usage}")
print(f"\n📌 choices[0] 구조:")
print(f"   .message.role     = {response.choices[0].message.role}")
print(f"   .message.content  = {response.choices[0].message.content[:50]}...")
print(f"   .finish_reason    = {response.choices[0].finish_reason}")
print(f"   .index            = {response.choices[0].index}")

print("\n✅ finish_reason 종류:")
print("   - 'stop': 정상 완료")
print("   - 'length': max_tokens 한도 도달")
print("   - 'content_filter': 콘텐츠 필터링")

---

## 2️⃣ 시스템 프롬프트와 역할 설정

시스템 프롬프트는 모델의 **페르소나, 행동 규칙, 출력 형식** 등을 설정합니다.

### 좋은 시스템 프롬프트의 요소

1. **역할 정의**: "당신은 ~입니다"
2. **행동 규칙**: "~해야 합니다", "~하지 마세요"
3. **출력 형식**: "JSON으로 응답하세요", "3줄 이내로"
4. **제약 조건**: "한국어로만", "전문 용어 사용"

---

In [ ]:
# 시스템 프롬프트 활용 예시들
print("=" * 60)
print("🎭 시스템 프롬프트 활용 예시")
print("=" * 60)

# 예시 1: 전문 번역가
print("\n--- 예시 1: 전문 번역가 ---")
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": """당신은 한영 전문 번역가입니다. 
규칙:
- 한국어가 입력되면 영어로 번역하세요.
- 영어가 입력되면 한국어로 번역하세요.
- 자연스러운 표현을 사용하세요.
- 번역 결과만 출력하세요. 다른 설명은 불필요합니다."""
        },
        {"role": "user", "content": "파인튜닝은 사전 학습된 모델을 특정 도메인에 맞게 추가 학습하는 과정입니다."}
    ],
    temperature=0.3,
    max_tokens=200,
)
print(f"🤖 번역 결과:\n{response1.choices[0].message.content}")

# 예시 2: 코드 리뷰어
print("\n--- 예시 2: 코드 리뷰어 ---")
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": """당신은 시니어 Python 개발자입니다.
규칙:
- 코드를 리뷰하고 개선점을 제안하세요.
- 한국어로 답변하세요.
- 형식: [장점], [개선점], [수정된 코드] 순서로 작성하세요."""
        },
        {"role": "user", "content": """다음 코드를 리뷰해주세요:

def get_data(url):
    import requests
    r = requests.get(url)
    data = r.json()
    return data"""}
    ],
    temperature=0.5,
    max_tokens=500,
)
print(f"🤖 코드 리뷰:\n{response2.choices[0].message.content}")

print("\n✅ 시스템 프롬프트로 모델의 역할과 행동을 제어할 수 있습니다!")

In [ ]:
# 시스템 프롬프트의 영향력 비교
print("=" * 60)
print("🔄 시스템 프롬프트 유무 비교")
print("=" * 60)

user_msg = "인공지능에 대해 설명해줘"

# 시스템 프롬프트 없이
print("\n📌 시스템 프롬프트 없이:")
resp_no_system = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": user_msg}
    ],
    max_tokens=150,
    temperature=0.7,
)
print(f"🤖 {resp_no_system.choices[0].message.content}")

# 초등학생 대상 시스템 프롬프트
print("\n📌 시스템 프롬프트: '초등학생에게 설명하듯이':")
resp_with_system = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "당신은 초등학생에게 과학을 가르치는 선생님입니다. 쉬운 비유와 예시를 사용하고, 짧고 재미있게 설명하세요."
        },
        {"role": "user", "content": user_msg}
    ],
    max_tokens=150,
    temperature=0.7,
)
print(f"🤖 {resp_with_system.choices[0].message.content}")

print("\n✅ 같은 질문이지만 시스템 프롬프트에 따라 응답 스타일이 완전히 다릅니다!")

---

## 3️⃣ 파라미터 조절 (temperature, top_p, max_tokens)

### 주요 파라미터

| 파라미터 | 범위 | 설명 |
|---------|------|------|
| `temperature` | 0~2 | 출력의 무작위성 조절. 0=결정적, 높을수록 창의적 |
| `top_p` | 0~1 | 누적 확률 기반 토큰 선택. 1=모든 토큰 후보 |
| `max_tokens` | 1~모델한도 | 최대 출력 토큰 수 |
| `frequency_penalty` | -2~2 | 반복 토큰 억제 |
| `presence_penalty` | -2~2 | 새로운 토픽 장려 |
| `n` | 1+ | 생성할 응답 수 |

### 권장 설정

| 사용 사례 | temperature | top_p |
|----------|-------------|-------|
| 코드 생성 | 0~0.3 | 0.1 |
| 데이터 추출 | 0 | 1 |
| 번역 | 0.3 | 1 |
| 일반 대화 | 0.7 | 1 |
| 창의적 글쓰기 | 0.9~1.2 | 0.95 |

⚠️ **주의**: `temperature`와 `top_p`를 동시에 조절하는 것은 권장하지 않습니다. 하나만 조절하세요.

---

In [ ]:
# Temperature 변화에 따른 출력 비교
print("=" * 60)
print("🌡️ Temperature 변화에 따른 출력 비교")
print("=" * 60)

prompt = "파이썬의 장점을 3가지 알려줘."
temperatures = [0, 0.3, 0.7, 1.0, 1.5]

for temp in temperatures:
    print(f"\n🌡️ Temperature = {temp}:")
    print("-" * 50)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temp,
        max_tokens=200,
    )
    print(f"🤖 {response.choices[0].message.content}")

print("\n" + "=" * 60)
print("💡 Temperature 0: 항상 같은 응답 (결정적)")
print("💡 Temperature 높음: 매번 다른 표현 (창의적이지만 불안정할 수 있음)")

In [ ]:
# max_tokens 효과 확인
print("=" * 60)
print("📏 max_tokens 효과 확인")
print("=" * 60)

prompt = "딥러닝의 역사를 설명해주세요."
max_tokens_list = [30, 100, 300]

for max_tok in max_tokens_list:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.5,
        max_tokens=max_tok,
    )
    answer = response.choices[0].message.content
    finish = response.choices[0].finish_reason
    used = response.usage.completion_tokens
    
    print(f"\n📏 max_tokens={max_tok} (실제 사용: {used}, 종료: {finish}):")
    print(f"🤖 {answer}")
    if finish == "length":
        print("⚠️ → 토큰 한도에 도달하여 잘렸습니다!")

print("\n✅ max_tokens가 부족하면 응답이 중간에 잘릴 수 있으니 주의하세요!")

---

## 4️⃣ 스트리밍 응답 처리

스트리밍을 사용하면 모델이 생성하는 토큰을 **실시간으로** 받아볼 수 있습니다.

### 스트리밍의 장점

- 🔸 **체감 응답 속도 향상**: 첫 토큰이 즉시 표시
- 🔸 **UX 개선**: ChatGPT처럼 글자가 타이핑되는 효과
- 🔸 **긴 응답 처리**: 전체 생성을 기다리지 않아도 됨

---

In [ ]:
# 스트리밍 응답 처리
import time

print("=" * 60)
print("🌊 스트리밍(Streaming) 응답")
print("=" * 60)

print("\n📝 질문: LLM 파인튜닝의 3단계를 설명해주세요.")
print("\n🤖 응답 (스트리밍):")
print("-" * 50)

start_time = time.time()
first_token_time = None
full_response = ""

# stream=True로 스트리밍 활성화
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "한국어로 간결하게 답변하세요."},
        {"role": "user", "content": "LLM 파인튜닝의 3단계를 설명해주세요."}
    ],
    temperature=0.7,
    max_tokens=500,
    stream=True,  # 스트리밍 활성화!
)

for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        content = chunk.choices[0].delta.content
        if first_token_time is None:
            first_token_time = time.time()
        print(content, end="", flush=True)
        full_response += content

end_time = time.time()

print("\n" + "-" * 50)
print(f"\n⏱️ 첫 토큰까지: {first_token_time - start_time:.2f}초")
print(f"⏱️ 전체 완료: {end_time - start_time:.2f}초")
print(f"📊 총 글자 수: {len(full_response)}")

### 🔥 스트리밍 vs 일반 응답 체감 비교

같은 질문을 **일반 응답**과 **스트리밍 응답**으로 나란히 비교해봅니다.  
스트리밍이 체감 속도에서 얼마나 차이나는지 직접 확인해보세요!

In [ ]:
# 스트리밍 vs 일반 응답 체감 비교
import time
from IPython.display import clear_output, display, Markdown

question = "Python의 장점 3가지를 간단히 알려주세요."
messages = [
    {"role": "system", "content": "한국어로 간결하게 답변하세요."},
    {"role": "user", "content": question}
]

# ===== 1. 일반 응답 (한번에 받기) =====
print("=" * 60)
print("1️⃣ 일반 응답 (stream=False) — 전체가 한번에 출력")
print("=" * 60)

start = time.time()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    max_tokens=200,
    stream=False,
)
elapsed = time.time() - start
print(f"\n{response.choices[0].message.content}")
print(f"\n⏱️ 대기 시간: {elapsed:.2f}초 (이 시간 동안 아무것도 안 보임!)")

# ===== 2. 스트리밍 응답 (토큰 단위로 받기) =====
print("\n" + "=" * 60)
print("2️⃣ 스트리밍 응답 (stream=True) — 글자가 타이핑되듯 출력")
print("=" * 60 + "\n")

start = time.time()
first_token = None
token_count = 0

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    max_tokens=200,
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        content = chunk.choices[0].delta.content
        if first_token is None:
            first_token = time.time()
        token_count += 1
        print(content, end="", flush=True)  # 한 글자씩 출력!

elapsed = time.time() - start
ttft = first_token - start if first_token else 0

print(f"\n\n⏱️ 첫 글자까지 (TTFT): {ttft:.2f}초 — 거의 즉시 시작!")
print(f"⏱️ 전체 완료: {elapsed:.2f}초")
print(f"📊 수신 chunk 수: {token_count}개")
print("\n💡 스트리밍은 전체 시간은 비슷하지만, 첫 글자가 빠르게 나와서 체감 속도가 훨씬 좋습니다!")


In [ ]:
# 스트리밍 vs 비스트리밍 응답 시간 비교
import time

print("=" * 60)
print("⏱️ 스트리밍 vs 비스트리밍 응답 시간 비교")
print("=" * 60)

test_prompt = "파이썬으로 간단한 웹 서버를 만드는 방법을 단계별로 설명해주세요."

# 비스트리밍
start = time.time()
response_no_stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": test_prompt}],
    max_tokens=300,
    stream=False,
)
no_stream_time = time.time() - start

# 스트리밍 (첫 토큰 시간 측정)
start = time.time()
stream_resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": test_prompt}],
    max_tokens=300,
    stream=True,
)
first_token = None
for chunk in stream_resp:
    if chunk.choices[0].delta.content and first_token is None:
        first_token = time.time() - start
stream_total_time = time.time() - start

print(f"\n📊 비스트리밍:")
print(f"   전체 응답 시간: {no_stream_time:.2f}초 (응답 전체가 한 번에 옴)")

print(f"\n📊 스트리밍:")
print(f"   첫 토큰 도착: {first_token:.2f}초 (사용자가 바로 글자를 볼 수 있음!)")
print(f"   전체 완료: {stream_total_time:.2f}초")

print(f"\n💡 스트리밍이 체감 속도가 {no_stream_time/first_token:.1f}배 빠릅니다!")
print("   → ChatGPT 같은 서비스가 스트리밍을 사용하는 이유!")

---

## 5️⃣ 멀티턴 대화 구현

OpenAI API는 **상태를 저장하지 않습니다(stateless)**. 

멀티턴 대화를 구현하려면 **이전 대화 내역을 매번 함께 보내야** 합니다.

```
1회차: [system, user1]                    → assistant1
2회차: [system, user1, assistant1, user2]  → assistant2
3회차: [system, user1, assistant1, user2, assistant2, user3] → assistant3
```

---

In [ ]:
# 멀티턴 대화 구현
print("=" * 60)
print("💬 멀티턴 대화 구현")
print("=" * 60)

# 대화 이력 관리
conversation = [
    {
        "role": "system",
        "content": "당신은 친절한 AI 프로그래밍 튜터입니다. 한국어로 답변하고, 초보자도 이해할 수 있게 설명합니다."
    }
]

def chat(user_message):
    """멀티턴 대화 함수"""
    # 사용자 메시지 추가
    conversation.append({"role": "user", "content": user_message})
    
    # API 호출 (전체 대화 이력 전송)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=conversation,
        temperature=0.7,
        max_tokens=300,
    )
    
    # 모델 응답 추출 및 대화 이력에 추가
    assistant_message = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": assistant_message})
    
    return assistant_message

# 대화 진행
print("\n👤 1턴: 파이썬에서 리스트와 튜플의 차이가 뭐야?")
print(f"🤖 {chat('파이썬에서 리스트와 튜플의 차이가 뭐야?')}")

print("\n" + "-" * 40)
print("\n👤 2턴: 그러면 딕셔너리는?")
print(f"🤖 {chat('그러면 딕셔너리는?')}")

print("\n" + "-" * 40)
print("\n👤 3턴: 이 세 가지를 언제 사용하면 좋을지 예시를 들어줘")
print(f"🤖 {chat('이 세 가지를 언제 사용하면 좋을지 예시를 들어줘')}")

print(f"\n📊 현재 대화 이력: {len(conversation)}개 메시지")
print("✅ 이전 대화 맥락을 기억하고 있는 것을 확인하세요!")

In [ ]:
# 대화 이력 구조 확인
print("=" * 60)
print("📋 대화 이력 구조 확인")
print("=" * 60)

for i, msg in enumerate(conversation):
    role_emoji = {"system": "⚙️", "user": "👤", "assistant": "🤖"}
    emoji = role_emoji.get(msg["role"], "❓")
    content_preview = msg["content"][:80] + "..." if len(msg["content"]) > 80 else msg["content"]
    print(f"\n{emoji} [{i}] role: {msg['role']}")
    print(f"   content: {content_preview}")

print(f"\n💡 매 요청마다 이 전체 이력이 API에 전송됩니다.")
print(f"💡 대화가 길어지면 토큰 수가 증가 → 비용 증가!")
print(f"💡 실무에서는 최근 N턴만 유지하거나 요약하는 전략을 사용합니다.")

---

## 6️⃣ JSON 모드와 구조화된 출력

API 응답을 **JSON 형태**로 받으면 프로그래밍에서 활용하기 훨씬 편리합니다.

### JSON 모드 사용법

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[...],
    response_format={"type": "json_object"},  # JSON 모드 활성화
)
```

⚠️ **주의**: JSON 모드 사용 시, 프롬프트에 "JSON으로 응답하세요"라는 지시를 포함해야 합니다.

---

In [ ]:
# JSON 모드 사용 예시
import json

print("=" * 60)
print("📋 JSON 모드 활용")
print("=" * 60)

# 예시 1: 감성 분석 결과를 JSON으로
print("\n--- 예시 1: 감성 분석 ---")
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": """텍스트의 감성을 분석하여 JSON으로 응답하세요.
형식: {"text": "원문", "sentiment": "positive/negative/neutral", "confidence": 0.0~1.0, "keywords": ["키워드1", "키워드2"]}"""
        },
        {"role": "user", "content": "이 제품 정말 훌륭합니다! 배송도 빠르고 품질도 최고예요."}
    ],
    temperature=0,
    response_format={"type": "json_object"},
)

result = json.loads(response.choices[0].message.content)
print(f"📊 분석 결과 (파싱된 JSON):")
print(json.dumps(result, ensure_ascii=False, indent=2))

# 프로그래밍적 활용
print(f"\n💡 프로그래밍 활용:")
print(f"   감성: {result.get('sentiment')}")
print(f"   신뢰도: {result.get('confidence')}")
print(f"   키워드: {result.get('keywords')}")

In [ ]:
# 예시 2: 구조화된 정보 추출
import json

print("=" * 60)
print("📋 구조화된 정보 추출")
print("=" * 60)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": """다음 텍스트에서 정보를 추출하여 JSON으로 응답하세요.
형식:
{
  "models": [
    {
      "name": "모델명",
      "organization": "개발사",
      "parameters": "파라미터 수",
      "release_year": 2024,
      "key_feature": "핵심 특징"
    }
  ]
}"""
        },
        {
            "role": "user",
            "content": """Meta는 2024년에 LLaMA 3를 출시했다. 8B와 70B 버전이 있으며, 
128K 컨텍스트 윈도우를 지원한다. Alibaba의 Qwen2.5는 0.5B부터 72B까지 다양한 크기로 
출시되었으며, 다국어 지원이 강점이다. Mistral AI의 Mistral 7B는 Apache 2.0 라이선스로 
공개되어 커뮤니티에서 인기가 높다."""
        }
    ],
    temperature=0,
    response_format={"type": "json_object"},
)

result = json.loads(response.choices[0].message.content)
print(f"\n📊 추출 결과:")
print(json.dumps(result, ensure_ascii=False, indent=2))

# 프로그래밍적 활용
print(f"\n💡 추출된 모델 수: {len(result.get('models', []))}개")
for m in result.get('models', []):
    print(f"   - {m.get('name')} ({m.get('organization')}): {m.get('key_feature')}")

print("\n✅ JSON 모드를 사용하면 LLM 출력을 프로그래밍에 바로 활용할 수 있습니다!")

---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 주제 | 핵심 내용 |
|------|----------|
| API 설정 | .env 파일로 안전하게 API Key 관리 |
| Chat Completions | messages 리스트에 role과 content로 대화 |
| 시스템 프롬프트 | 모델의 역할/규칙/출력 형식 설정 |
| 파라미터 | temperature(창의성), max_tokens(길이), top_p(후보 범위) |
| 스트리밍 | stream=True로 실시간 응답, 체감 속도 향상 |
| 멀티턴 대화 | 대화 이력을 매번 함께 전송 (stateless) |
| JSON 모드 | response_format으로 구조화된 출력 |

### API 비용 절약 팁

- 💡 개발 중에는 `gpt-4o-mini` 사용 (가장 저렴)
- 💡 `max_tokens`를 적절히 제한
- 💡 `temperature=0`으로 캐싱 효과 (동일 입력 → 동일 출력)
- 💡 멀티턴 대화 시 이력을 적절히 정리

### 다음 세션 예고

- 🔜 LangChain 프레임워크 소개
- 🔜 프롬프트 템플릿, LCEL 파이프라인, Memory

---

### 💡 실습 과제

1. 자신만의 시스템 프롬프트를 작성하여 특정 역할을 하는 AI를 만들어보세요.
2. temperature를 0과 1.5로 설정하여 같은 질문에 대한 응답 차이를 비교해보세요.
3. JSON 모드를 활용하여 뉴스 기사에서 핵심 정보를 추출하는 코드를 작성해보세요.
4. (선택) 멀티턴 대화에서 최근 5턴만 유지하도록 코드를 수정해보세요.

---